In [1]:
    library(Seurat)
    library(MAST)
    library(dplyr)
    library(data.table)
    library(ggplot2)
    library(schard)
    library(tibble)
    library(stringr)


Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built under R 4.2.3 but the current version is
4.4.3; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed

‘SeuratObject’ was built with package ‘Matrix’ 1.6.5 but the current
version is 1.7.3; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMad

In [ ]:
seo = schard::h5ad2seurat('/data1st2/junyi/output/atac1112/subset/region_nt/HIP_HIP_GABA.h5ad')
df_qc <- read.csv('/data2st1/junyi/output/atac0627/frac_qc.csv', row.names = 1)
metadata <- seo@meta.data
cell_groups <- unique(metadata[["celltype.L2"]])
metadata$cell_bc <- rownames(metadata)
meta2 <- merge(metadata, df_qc, by = "sample", all.x = TRUE)
rownames(meta2) <- meta2$cell_bc
seo@meta.data <- meta2

In [89]:
cell_group = cell_groups[1]

In [90]:
seo_subset = subset(seo, subset = celltype.L2 == cell_group)
# Filter genes expressed in at least 5 cells
exprs <- GetAssayData(seo_subset, slot = "counts")
peaks_to_keep <- rowSums(exprs > 0) >= 10
#seo_subset2 <- seo_subset[genes_to_keep, ]
seo_subset2 <- seo_subset[peaks_to_keep, ]

In [99]:
# rename column Fraction.of.high.quality.fragments.overlapping.peak to frac_peak
colnames(seo_subset2@meta.data)[colnames(seo_subset2@meta.data) == "Fraction.of.high.quality.fragments.overlapping.peaks"] <- "frac_peak"

In [100]:
colnames(seo_subset2@meta.data)

[1] "sample"                    "orig.ident"               
 [3] "nCount_RNA"                "nFeature_RNA"             
 [5] "_index"                    "doublet_probability"      
 [7] "doublet_score"             "leiden"                   
 [9] "leiden_default"            "leiden_res_0.1"           
[11] "leiden_res_0.2"            "leiden_res_0.3"           
[13] "leiden_res_0.4"            "leiden_res_0.5"           
[15] "leiden_res_0.6"            "leiden_res_0.7"           
[17] "leiden_res_0.8"            "leiden_res_0.9"           
[19] "leiden_res_1.0"            "leiden_res_1.1"           
[21] "leiden_res_1.2"            "leiden_res_1.3"           
[23] "leiden_res_1.4"            "leiden_res_1.5"           
[25] "leiden_res_1.6"            "leiden_res_1.7"           
[27] "leiden_res_1.8"            "leiden_res_1.9"           
[29] "celltype.L2.Condition"     "celltype.L1"              
[31] "celltype.L2"               "Neurotransmitter_celltype"
[33] "celltype.L1_ct"            "Sample_name"              
[35] "Condition"                 "Region"                   
[37] "celltype.L2.raw"           "region_nt"                
[39] "celltype.L3"               "celltype.L4"              
[41] "celltype.L2.refined"       "expriment"                
[43] "cell_bc"                   "frac_peak"                
[45] "farcq"                     "date"

In [107]:
perform_mast_analysis <- function(seurat_obj,
                                  group.by, 
                                  compare.by,
                                  group1, 
                                  group2, 
                                  batch.by,
                                  freq_expressed = 0.1,
                                  save.as.tmp = TRUE) {
  library(MAST)
  library(data.table)

  
  expr_matrix <- GetAssayData(seurat_obj, layer = "data")
      
  #filter_genes <- !grepl("^Rp", rownames(expr_matrix)) & !grepl("^mt-", rownames(expr_matrix))
  #expr_matrix <- expr_matrix[filter_genes, ]
    
  metadata <- seurat_obj@meta.data
  cell_groups <- unique(metadata[[group.by]])
  
  all_results <- data.frame()
  
  for (cell_group in cell_groups) {
    tryCatch({
      cat("====================================\n")
      cat("Analyzing cell group:", cell_group, "\n")
      
      cells_in_group <- rownames(metadata[metadata[[group.by]] == cell_group, ])
      cells_group1 <- cells_in_group[metadata[cells_in_group, compare.by] == group1]
      cells_group2 <- cells_in_group[metadata[cells_in_group, compare.by] == group2]
      
      if(length(cells_group1) == 0 || length(cells_group2) == 0) {
        cat("  Skipped: one of the treatment groups has zero cells\n")
        next
      }
      
      selected_cells <- c(cells_group1, cells_group2)
      dat.tmp <- expr_matrix[, selected_cells]
      anno.tmp <- metadata[selected_cells, ]
      
      # 打印batch和treatment分布表
      batch_treatment_table <- table(anno.tmp[[batch.by]], anno.tmp[[compare.by]])
      cat("  Batch x Treatment distribution:\n")
      print(batch_treatment_table)
      
      # 判断每个batch是否包含两个treatment组
      batches_all_have_both_treatments <- all(rowSums(batch_treatment_table > 0) == ncol(batch_treatment_table))
      
      # 是否有多个batch
      multiple_batches_exist <- nrow(batch_treatment_table) > 1
      
      # 根据分布决定是否用batch效应
      if (multiple_batches_exist && batches_all_have_both_treatments) {
        use_batch_effect <- TRUE
        cat("  Will use batch effect in model\n")
      } else {
        use_batch_effect <- FALSE
        if(!multiple_batches_exist) {
          cat("  Only one batch found - batch effect not used\n")
        } else if (!batches_all_have_both_treatments) {
          cat("  Not all batches contain both treatments - batch effect not used\n")
        }
      }
      use_batch_effect <- FALSE
      # 创建MAST对象
      sca <- FromMatrix(as.matrix(dat.tmp), cData = anno.tmp)
      
      # 基因表达频率过滤
      select.genes <- freq(sca) > freq_expressed
      sca <- sca[select.genes, ]
      
      # 设定compare_group factor
      colData(sca)$compare_group <- factor(colData(sca)[[compare.by]], levels = c(group1, group2))
      
      # add ngenes
      cdr2 <- colSums(assay(sca) > 0)
      colData(sca)$ngeneson <- scale(cdr2)

      if (use_batch_effect) {
        colData(sca)$batch <- factor(colData(sca)[[batch.by]])
        zlm_model <- zlm(~ compare_group + batch + ngeneson + frac_peak, sca)
      } else {
        zlm_model <- zlm(~ compare_group + ngeneson + frac_peak, sca)
      }
      
      # 差异分析
      contrast_name <- paste0("compare_group", group2)
      summary_result <- summary(zlm_model, doLRT = contrast_name)
      summaryDt <- summary_result$datatable
      
      fcHurdle <- merge(
        summaryDt[contrast == contrast_name & component == 'H', .(primerid, `Pr(>Chisq)`)],
        summaryDt[contrast == contrast_name & component == 'logFC', .(primerid, coef, ci.hi, ci.lo)],
        by = 'primerid'
      )
      if(nrow(fcHurdle) == 0){
        cat("  Warning: no genes passed filtering\n")
        next
      }
      
      fcHurdle[, padjust := p.adjust(`Pr(>Chisq)`, 'bonferroni')]
      fcHurdle[[group.by]] <- cell_group
      
      all_results <- rbind(all_results, fcHurdle)
      
      if(save.as.tmp){
        save(fcHurdle, file = paste0("tmp.degene.result.", make.names(cell_group), ".rda"))
        write.csv(fcHurdle, file = paste0("tmp.degene.result.", make.names(cell_group), ".csv"))
      }
    }, error = function(e){
      cat("  Error in cell group ", cell_group, " : ", e$message, "\n")
    })
  }
  
  if (nrow(all_results) == 0) {
    warning("No differential expression results were generated.")
  }
  
  return(all_results)
}


In [104]:
outfolder <- '/data2st2/junyi/output/test/dar/'
setwd(outfolder)
print(outfolder)


[1] "/data2st2/junyi/output/test/dar/"


In [110]:
# check if data is normalized
max_val_counts <- max(GetAssayData(seo_subset2, slot = "counts"))
max_val_data   <- max(GetAssayData(seo_subset2, slot = "data"))


In [108]:
r.deg_M <- perform_mast_analysis(seo_subset2,
                                group.by = "celltype.L2",
                                compare.by = "expriment",
                                group1 = "MW",
                                group2 = "MC",
                            batch.by = "date")

Analyzing cell group: HPF_Lamp5_GABA 
  Batch x Treatment distribution:
          
            MC  MW
  20250123  42 222
  20250305 146  70
  Will use batch effect in model


`cData` has no wellKey.  I'll make something up.

Assuming data assay in position 1, with name et is log-transformed.


Done!

Combining coefficients and standard errors

Calculating log-fold changes

Calculating likelihood ratio tests

Refitting on reduced model...


Done!



In [106]:
r.deg_M

primerid,Pr(>Chisq),coef,ci.hi,ci.lo,padjust,celltype.L2
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
chr10:100487209-100487710,0.89951476,-0.014641011,0.08421736,-0.11349938,1,HPF_Lamp5_GABA
chr10:100588995-100589496,0.24616410,-0.022270598,0.01452163,-0.05906282,1,HPF_Lamp5_GABA
chr10:10107158-10107659,0.87579479,-0.006702653,0.04637701,-0.05978232,1,HPF_Lamp5_GABA
chr10:10321040-10321541,0.76882126,0.016877101,0.07406136,-0.04030716,1,HPF_Lamp5_GABA
chr10:103367425-103367926,0.45727044,-0.048761455,0.03900006,-0.13652297,1,HPF_Lamp5_GABA
chr10:104595839-104596340,0.84672817,-0.020853765,0.05027613,-0.09198366,1,HPF_Lamp5_GABA
chr10:105574236-105574737,0.99191192,-0.002252339,0.03770198,-0.04220666,1,HPF_Lamp5_GABA
chr10:105841143-105841644,0.46272578,-0.039515539,0.03777737,-0.11680845,1,HPF_Lamp5_GABA
chr10:105992410-105992911,0.22413580,0.007178447,0.02975318,-0.01539629,1,HPF_Lamp5_GABA
